# Variational Quantum Classifier (VQC) — Iris Dataset

This notebook provides an interactive exploration of the Variational Quantum Classifier implementation.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('..')

try:
    import pennylane as qml
    print("✅ PennyLane imported successfully")
except ImportError:
    print("⚠️ PennyLane not available, will use NumPy simulator")

from vqc_iris import (
    circuit, N_QUBITS, N_LAYERS, BATCH_SIZE, LEARNING_RATE, N_EPOCHS
)

print("\n📊 Configuration:")
print(f"  Qubits: {N_QUBITS}")
print(f"  Layers: {N_LAYERS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Epochs: {N_EPOCHS}")

## 2. Load and Explore Iris Dataset

In [ ]:
# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

print(f"📊 Dataset Shape: {X.shape}")
print(f"   Classes: {len(np.unique(y))}")
print(f"   Features per sample: {X.shape[1]}")
print(f"\n   Feature Names: {iris.feature_names}")
print(f"   Target Names: {iris.target_names}")
print(f"\n   Class Distribution: {np.bincount(y)}")

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
feature_names = iris.feature_names

for idx, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    for class_id in range(3):
        mask = y == class_id
        ax.hist(X[mask, idx], alpha=0.5, label=iris.target_names[class_id])
    ax.set_xlabel(name)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

print("✅ Feature distributions plotted")

## 3. Data Preprocessing

In [ ]:
# Split into train/val/test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} samples")
print(f"Val:   {len(X_val)} samples")
print(f"Test:  {len(X_test)} samples")

In [ ]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Normalize to [-1, 1] for quantum encoding
X_train_norm = (X_train - X_train.min()) / (X_train.max() - X_train.min()) * 2 - 1
X_val_norm = (X_val - X_val.min()) / (X_val.max() - X_val.min()) * 2 - 1
X_test_norm = (X_test - X_test.min()) / (X_test.max() - X_test.min()) * 2 - 1

print("✅ Normalization complete")
print(f"\nTrain data range: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")
print(f"Val data range:   [{X_val_norm.min():.3f}, {X_val_norm.max():.3f}]")
print(f"Test data range:  [{X_test_norm.min():.3f}, {X_test_norm.max():.3f}]")

## 4. Quantum Circuit Visualization

In [ ]:
# Initialize parameters
np.random.seed(42)
params = np.random.randn(N_LAYERS, N_QUBITS, 2) * 0.1

print(f"📊 Parameter Shape: {params.shape}")
print(f"   Total trainable parameters: {np.prod(params.shape)}")
print(f"\n   Parameter ranges:")
print(f"   Min: {params.min():.4f}")
print(f"   Max: {params.max():.4f}")
print(f"   Mean: {params.mean():.4f}")
print(f"   Std: {params.std():.4f}")

In [ ]:
# Test quantum circuit with sample input
sample_input = X_train_norm[0]
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample input: {sample_input}")

try:
    # Try to run circuit using PennyLane
    output = circuit(sample_input, params)
    print(f"\n✅ Circuit output (Z expectations): {output}")
    print(f"   Output range: [{np.min(output):.4f}, {np.max(output):.4f}]")
except Exception as e:
    print(f"⚠️ Could not run circuit: {e}")
    print("   This may be expected if PennyLane is not installed")

## 5. Training Example (Mini Batch)

In [ ]:
# Show how batches would be created
n_batches = len(X_train_norm) // BATCH_SIZE
print(f"📊 Batch Configuration:")
print(f"   Total samples: {len(X_train_norm)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Number of batches: {n_batches}")
print(f"   Samples per epoch: {n_batches * BATCH_SIZE}")

# Show first batch
X_batch = X_train_norm[:BATCH_SIZE]
y_batch = y_train[:BATCH_SIZE]

print(f"\n📦 First Batch:")
print(f"   Shape: {X_batch.shape}")
print(f"   Classes: {np.unique(y_batch)}")
print(f"   Class counts: {np.bincount(y_batch)}")

## 6. Cross-Entropy Loss Function

In [ ]:
# Demonstrate cross-entropy loss
def compute_loss(probs, labels):
    """Compute cross-entropy loss"""
    batch_size = probs.shape[0]
    log_probs = np.log(np.maximum(probs, 1e-10))
    loss = -np.mean(log_probs[np.arange(batch_size), labels])
    return loss

# Example distributions
perfect_probs = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

random_probs = np.ones((3, 3)) / 3

labels = np.array([0, 1, 2])

perfect_loss = compute_loss(perfect_probs, labels)
random_loss = compute_loss(random_probs, labels)

print(f"🔍 Cross-Entropy Loss Examples:")
print(f"   Perfect predictions: {perfect_loss:.6f}")
print(f"   Random predictions:  {random_loss:.6f}")
print(f"   Expected random (ln(3)): {np.log(3):.6f}")

## 7. Run Full Training

In [ ]:
# Note: This will take time. Uncomment to run full training.
# from vqc_iris import train_vqc, evaluate_model, plot_training_curves, plot_confusion_matrix
# 
# params_trained, history = train_vqc(X_train_norm, y_train, X_val_norm, y_val)
# y_pred, accuracy = evaluate_model(X_test_norm, y_test, params_trained)
# 
# plot_training_curves(history, Path('results'))
# plot_confusion_matrix(y_test, y_pred, Path('results'))

print("💡 To run full training, execute vqc_iris.py directly:")
print("   python vqc_iris.py")
print("\nOr uncomment the code above in this notebook.")

## 8. Summary

In [ ]:
print("""\n🔬 VARIATIONAL QUANTUM CLASSIFIER SUMMARY
============================================

📊 Dataset:
   - 150 Iris samples, 4 features, 3 classes
   - Split: 90 train, 30 val, 30 test

⚛️  Quantum Circuit:
   - Qubits: 4
   - Layers: 3
   - Trainable Parameters: 24

🧠 Hybrid Algorithm:
   1. Feature Encoding: Angle embedding (RY rotations)
   2. Ansatz: RY/RZ rotations + CNOT entanglement
   3. Measurement: Pauli-Z expectations
   4. Optimization: Finite-difference gradient descent

📈 Training:
   - Learning Rate: 0.08
   - Epochs: 80
   - Batch Size: 16
   - Loss: Cross-entropy

✅ Expected Results:
   - Test Accuracy: ~83.3%
   - Perfect Setosa classification (linearly separable)
   - Reasonable Versicolor/Virginica boundary learning
""")